# Day 7 Lab: Profile and Optimize AI Kernels

## Mission
Predict the cost of an AI workload, measure representative kernels, classify bottlenecks, and improve one workload using evidence.

**Required:** Python, NumPy, Pandas, Matplotlib  
**Optional:** PyTorch and CUDA-capable GPU

> This lab focuses on single-device performance engineering as preparation for Day 8.

## Learning objectives

After the lab, you should be able to:

1. Estimate model parameters, parameter memory, training FLOPs, and ideal runtime.
2. Benchmark kernels using warm-up, repeated trials, and robust summaries.
3. Estimate bytes moved, arithmetic intensity, achieved FLOP/s, and achieved bandwidth.
4. Classify kernels as compute-, bandwidth-, or overhead-bound using simplified Roofline reasoning.
5. Measure sequence-length scaling in attention-score computation.
6. optimize one workload and justify the result with measurements.

## Part 0: Environment and reproducibility

Run the setup cell. Record the hardware and library versions in your final memo.

In [ ]:
import os
import sys
import time
import math
import platform
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())

try:
    import torch
    TORCH_AVAILABLE = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("PyTorch:", torch.__version__)
    print("PyTorch device:", DEVICE)
    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
except Exception:
    TORCH_AVAILABLE = False
    DEVICE = "cpu"
    print("PyTorch not available; NumPy path will be used.")

## Part 1: Build an AI cost calculator

Use these simplified planning models:

- Dense transformer block parameters: `12 × layers × hidden_size²`
- Token embeddings: `vocab_size × hidden_size`
- Approximate training FLOPs: `6 × parameters × tokens`
- Ideal training time: `training_FLOPs / sustained_FLOPs_per_second`
- Attention-score elements per layer: `batch × heads × sequence²`

These are planning approximations, not exact profiler results.

### Task 1.1
Complete the functions.

In [ ]:
def transformer_parameter_estimate(num_layers, hidden_size, vocab_size, tie_embeddings=True):
    # TODO: estimate block and embedding parameters.
    block_params = None
    embedding_params = None
    output_params = 0 if tie_embeddings else None
    total = None
    return {
        "block_params": block_params,
        "embedding_params": embedding_params,
        "output_params": output_params,
        "total_params": total,
    }


def parameter_memory_gb(num_params, bytes_per_param):
    # Use decimal GB for this lab.
    # TODO
    pass


def training_flops(num_params, num_tokens):
    # TODO
    pass


def ideal_training_days(total_flops, sustained_tflops):
    # TODO: convert sustained TFLOP/s to FLOP/s and seconds to days.
    pass


def attention_score_memory_gb(batch, heads, sequence, bytes_per_element=2):
    # TODO: calculate one attention-score tensor.
    pass

### Task 1.2: Model comparison

Estimate the following designs:

| Design | Layers | Hidden size | Vocabulary | Tokens |
|---|---:|---:|---:|---:|
| Small | 12 | 768 | 32,000 | 10B |
| Medium | 24 | 1,024 | 50,000 | 20B |
| Large | 32 | 2,048 | 50,000 | 60B |

For each design, calculate:

- Parameters
- BF16 parameter memory
- Approximate training FLOPs
- Ideal days at 150 sustained TFLOP/s
- BF16 attention-score memory for batch 2, 16 heads, sequence 4,096

Create a DataFrame.

In [ ]:
# TODO: create the model comparison DataFrame.

### Task 1.3: Scaling questions

Answer in a Markdown cell:

1. What happens to block parameters when hidden size doubles?
2. What happens to attention-score memory when sequence length doubles?
3. Why can a model's parameters fit while training still fails from insufficient memory?
4. Why is ideal runtime usually optimistic?

## Part 2: Implement a reliable benchmark harness

A useful benchmark should:

- Warm up the operation
- Run multiple trials
- Synchronize CUDA before and after timing, if applicable
- Report median and variability
- Keep setup outside the timed function
- Verify correctness separately

### Task 2.1
Complete `benchmark`.

In [ ]:
def synchronize_if_needed():
    if TORCH_AVAILABLE and DEVICE == "cuda":
        torch.cuda.synchronize()


def benchmark(fn, warmups=5, trials=15):
    # TODO: run warmups, then repeated timed trials.
    # Return median, minimum, maximum, mean, standard deviation, and all times.
    pass

### Task 2.2: Validate the harness

Benchmark a function that sleeps for roughly 5 milliseconds. The median should be close to, but not necessarily exactly, 5 ms.

In [ ]:
# TODO: benchmark time.sleep(0.005) and display the result in milliseconds.

## Part 3: Benchmark representative AI kernels

The lab uses NumPy by default so it works on laptops. If a CUDA GPU is available, the optional extension repeats selected experiments with PyTorch.

Benchmark:

1. Vector addition
2. ReLU
3. Matrix-vector multiplication
4. Matrix-matrix multiplication
5. Batched matrix multiplication

### Task 3.1: Create inputs

Choose sizes that complete quickly but are large enough to measure. Suggested starting values:

- Vector length: 5,000,000
- Matrix-vector: 4,096 × 4,096
- Matrix-matrix: 1,024 × 1,024
- Batched matrix multiplication: batch 16, matrices 256 × 256

Reduce sizes if necessary.

In [ ]:
# TODO: create FP32 NumPy inputs and kernel callables.
# Keep allocation outside timed functions.

### Task 3.2: Verify correctness

Before timing, verify output shapes and compare at least one result against an independently expressed calculation.

In [ ]:
# TODO: correctness checks using assert and np.allclose.

### Task 3.3: Measure runtime

Benchmark each kernel and create a table with median milliseconds and variability.

In [ ]:
# TODO: benchmark the five kernels and create a DataFrame.

## Part 4: FLOPs, bytes, arithmetic intensity, and achieved rates

Use these simplified counts.

### Vector addition, length `n`

- FLOPs: `n`
- Bytes: `3 × n × 4`

### ReLU, length `n`

Use a simplified count:

- Operations: `n`
- Bytes: `2 × n × 4`

### Matrix-vector, `m × k`

- FLOPs: `2mk`
- Minimum bytes: `(mk + k + m) × 4`

### Matrix-matrix, `(m × k) @ (k × n)`

- FLOPs: `2mkn`
- Minimum bytes: `(mk + kn + mn) × 4`

### Batched matrix multiplication

Multiply the matrix-matrix counts by batch size.

### Task 4.1
Add columns for:

- Approximate FLOPs
- Approximate bytes
- Arithmetic intensity
- Achieved GFLOP/s
- Achieved GB/s

In [ ]:
# TODO: create analytical counts and join them with timing results.

### Task 4.2: Interpret

Rank kernels by arithmetic intensity. Then answer:

1. Which kernel is most likely bandwidth-bound?
2. Which kernel is most likely compute-bound?
3. Which result is most affected by Python or call overhead?
4. Why are the byte estimates lower bounds for matrix operations?

## Part 5: Simplified Roofline analysis

First estimate sustained memory bandwidth with a large copy operation. This is not a perfect hardware bandwidth benchmark, but it provides a local reference.

### Task 5.1: Approximate memory bandwidth

Time `destination[:] = source`. Count one read and one write.

In [ ]:
# TODO: benchmark a large array copy and estimate GB/s.

### Task 5.2: Estimate a compute ceiling

Use the highest achieved GFLOP/s among matrix operations as a conservative empirical compute ceiling.

For each kernel:

`roofline_GFlop_s = min(empirical_compute_ceiling, bandwidth_GB_s × intensity)`

Then calculate:

`roofline_efficiency = achieved_GFlop_s / roofline_GFlop_s`

In [ ]:
# TODO: add roofline ceiling, predicted regime, and efficiency columns.

### Task 5.3: Plot a simplified Roofline chart

Use log scales:

- x-axis: arithmetic intensity
- y-axis: performance in GFLOP/s
- draw the bandwidth line
- draw the compute ceiling
- annotate the five kernels

In [ ]:
# TODO: create the Roofline plot.

## Part 6: Overhead and batching experiment

Compare many small matrix multiplications with one batched matrix multiplication.

### Task 6.1
Use 128 matrices of size 64 × 64.

- Method A: Python loop calling `@` 128 times
- Method B: one batched `np.matmul`

Verify that outputs match, then benchmark both.

In [ ]:
# TODO: implement loop and batched versions, check correctness, and benchmark.

### Task 6.2: Analyze

Calculate speedup and explain it using:

- Python loop overhead
- Library-call overhead
- Larger operations and hardware utilization
- Memory costs of batching

## Part 7: Attention sequence-length scaling

Measure only the attention-score multiplication, not a full attention layer.

Let:

- batch = 1
- heads = 4
- head dimension = 64
- sequence lengths = 128, 256, 512, 1,024, and optionally 2,048

Calculate:

`score = Q @ K.transpose(0, 1, 3, 2)`

### Task 7.1
Benchmark runtime, estimate output memory, and record `runtime / sequence²`.

In [ ]:
# TODO: run sequence-length experiments. Catch MemoryError and stop gracefully.

### Task 7.2: Plot and interpret

Create:

1. Runtime versus sequence length
2. Score-tensor memory versus sequence length
3. Runtime versus sequence length squared

Answer:

- Does measured behavior resemble quadratic scaling?
- Where do fixed overheads appear?
- At which size does memory become concerning?

In [ ]:
# TODO: create the three plots.

## Part 8: Optimization challenge

Choose one:

### Track A: Vectorization
Compare a Python loop with NumPy vectorized elementwise computation.

### Track B: Batching
Vary batch size for small matrix multiplications.

### Track C: Fusion concept
Compare a sequence of elementwise expressions that creates intermediates with an in-place or reduced-intermediate version, while preserving correctness.

### Track D: Precision
If your hardware supports it reliably, compare FP32 with a lower precision. Report numerical error as well as runtime and memory.

Your optimization must include:

- Prediction
- Baseline measurement
- Modified implementation
- Correctness check
- Repeated measurement
- Speedup
- Explanation

In [ ]:
# TODO: implement the selected optimization track.

## Part 9: Performance engineering memo

Complete this section in Markdown.

### Environment

- Processor or GPU:
- Python and library versions:
- Precision:

### Workload and prediction

What operation did you optimize? Was it expected to be compute-, bandwidth-, or overhead-bound?

### Evidence

Report:

- Median runtime
- Variability
- Estimated FLOPs
- Estimated bytes
- Arithmetic intensity
- Achieved GFLOP/s or GB/s

### Optimization and result

What changed? What was the speedup?

### Interpretation

Why did the optimization work or fail?

### Limitations

Identify at least three limitations of this lab's models or measurements.

### Bridge to Day 8

Which remaining constraint might require tensor parallelism, pipeline parallelism, ZeRO, or checkpointing?

## Submission checklist

- [ ] Cost calculator and model comparison
- [ ] Benchmark harness
- [ ] Five-kernel benchmark table
- [ ] Arithmetic-intensity and achieved-rate analysis
- [ ] Roofline plot
- [ ] Batching experiment
- [ ] Attention-scaling plots
- [ ] Optimization challenge
- [ ] Performance engineering memo